In [ ]:
# Generate and freeze train/test splits for TCGA-BRCA survival.
# Run this ONCE.  All downstream notebooks load from the saved CSVs.
#
# Outputs (in ../splits/):
#   brca_survival_sample_ids.csv   — frozen sample ordering (intersection)
#   brca_survival_train_idx.csv    — 0-based row indices into sample_ids
#   brca_survival_test_idx.csv
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
matrices_dir = Path("../matrices")
data_dir     = Path("../data")
splits_dir   = Path("../splits")
splits_dir.mkdir(exist_ok=True, parents=True)

SPLIT_TAG    = "brca_survival"
TEST_FRAC    = 0.20
RANDOM_STATE = 42

In [ ]:
# Load sample IDs from each modality matrix (index column only)
rna_ids   = pd.read_csv(matrices_dir / "RNA_X_full.csv",         usecols=[0]).iloc[:, 0].values
meth_ids  = pd.read_csv(matrices_dir / "Methylation_X_full.csv", usecols=[0]).iloc[:, 0].values
cnv_ids   = pd.read_csv(matrices_dir / "CNV_X_full.csv",         usecols=[0]).iloc[:, 0].values

print(f"RNA  : {len(rna_ids)} samples")
print(f"Meth : {len(meth_ids)} samples")
print(f"CNV  : {len(cnv_ids)} samples")

In [ ]:
# Load survival data and clean response
surv_raw = pd.read_csv(data_dir / "BRCA_survival.tsv", sep="\t", index_col=0)
surv_df  = surv_raw[["OS", "OS.time"]].copy()
surv_df.columns = ["event", "time"]
surv_df["event"] = pd.to_numeric(surv_df["event"], errors="coerce")
surv_df["time"]  = pd.to_numeric(surv_df["time"],  errors="coerce")

n_raw = len(surv_df)

# Drop any row where OS or OS.time is missing
surv_df = surv_df.dropna(subset=["event", "time"])
n_after_na = len(surv_df)
print(f"Dropped {n_raw - n_after_na} rows with missing OS / OS.time  "
      f"({n_raw} → {n_after_na})")

# Drop rows with non-positive survival time (Cox requires time > 0)
mask_pos = surv_df["time"] > 0
n_zero   = (~mask_pos).sum()
if n_zero > 0:
    print(f"Dropped {n_zero} rows with OS.time ≤ 0  "
          f"(times: {surv_df.loc[~mask_pos, 'time'].tolist()})")
surv_df = surv_df[mask_pos]

# Validate event is binary {0, 1}
bad_event = surv_df["event"].dropna()
assert set(bad_event.unique()).issubset({0.0, 1.0}), \
    f"Unexpected event values: {bad_event.unique()}"

print(f"Final survival rows: {len(surv_df)}  |  "
      f"events: {int(surv_df['event'].sum())}  "
      f"({100*surv_df['event'].mean():.1f}%)")
print(f"OS.time  min={surv_df['time'].min():.0f}  "
      f"median={surv_df['time'].median():.0f}  "
      f"max={surv_df['time'].max():.0f}")

In [ ]:
# Compute frozen intersection and sort for reproducibility
common = sorted(
    set(rna_ids) & set(meth_ids) & set(cnv_ids) & set(surv_df.index)
)
print(f"Common samples (all modalities + valid survival): {len(common)}")

# Final response arrays — guaranteed non-null, time > 0
event_all = surv_df.loc[common, "event"].values.astype(np.float32)
time_all  = surv_df.loc[common, "time"].values.astype(np.float32)

assert not np.any(np.isnan(event_all)), "NaN in event after filtering"
assert not np.any(np.isnan(time_all)),  "NaN in time after filtering"
assert np.all(time_all > 0),            "Non-positive time after filtering"

print(f"Events: {int(event_all.sum())} / {len(common)}  ({100*event_all.mean():.1f}%)")

In [ ]:
# Stratified 80/20 split (stratify on event to preserve event rate)
idx_all = np.arange(len(common))
idx_tr, idx_te = train_test_split(
    idx_all,
    test_size=TEST_FRAC,
    random_state=RANDOM_STATE,
    stratify=event_all.astype(int),
)

print(f"Train: {len(idx_tr)}  ({int(event_all[idx_tr].sum())} events, "
      f"{100*event_all[idx_tr].mean():.1f}%)")
print(f"Test : {len(idx_te)}  ({int(event_all[idx_te].sum())} events, "
      f"{100*event_all[idx_te].mean():.1f}%)")

In [ ]:
# Save — warn if overwriting
sid_path = splits_dir / f"{SPLIT_TAG}_sample_ids.csv"
trn_path = splits_dir / f"{SPLIT_TAG}_train_idx.csv"
tst_path = splits_dir / f"{SPLIT_TAG}_test_idx.csv"

for p in [sid_path, trn_path, tst_path]:
    if p.exists():
        print(f"WARNING: {p.name} already exists — overwriting.")

pd.DataFrame({"sample_id": common}).to_csv(sid_path, index=False)
pd.DataFrame({"index": idx_tr}).to_csv(trn_path,   index=False)
pd.DataFrame({"index": idx_te}).to_csv(tst_path,   index=False)

print(f"Saved:")
print(f"  {sid_path}")
print(f"  {trn_path}")
print(f"  {tst_path}")

In [ ]:
# Verify round-trip
sid_check = pd.read_csv(sid_path)["sample_id"].tolist()
tr_check  = pd.read_csv(trn_path)["index"].values
te_check  = pd.read_csv(tst_path)["index"].values

assert sid_check == common,                                "sample_ids mismatch"
assert set(tr_check) | set(te_check) == set(idx_all),     "index union mismatch"
assert len(set(tr_check) & set(te_check)) == 0,           "train/test overlap"
print("Round-trip verification passed.")